# Training YOLO26n-Depth di Google Colab

Notebook ini mandiri: kode arsitektur dan pipeline training ada di dalam sel. Dataset SUN RGB-D dibaca dari Google Drive; path dataset dan output bisa disesuaikan pada sel konfigurasi.

1. Pilih runtime GPU di Colab.
2. Jalankan sel dari atas ke bawah.
3. Pastikan `SUN_ROOT` menunjuk ke folder dataset SUN RGB-D yang sudah diekstrak di Drive.

Notebook ini melatih dengan SUN RGB-D saja. COCO tidak diunduh, sehingga tidak bergantung pada host COCO yang gagal verifikasi sertifikat.

In [1]:
from google.colab import drive
from pathlib import Path
import threading

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
SUN_ROOT = DRIVE_ROOT / 'datasets/SUNRGBD'  # ubah sesuai folder dataset hasil ekstraksi
RUN_DIR = DRIVE_ROOT / 'comvis_runs/arsitektur'
RUN_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ('person', 'laptop', 'cell phone', 'bottle')
IMAGE_SIZE = (480, 640)  # tinggi, lebar
DEPTH_MAX_M = 10.0
EPOCHS = 50
BATCH_SIZE = 8
NUM_WORKERS = 2
SEED = 42
RESUME_FROM = RUN_DIR / 'best.pt'  # set None untuk mulai dari awal

print('SUN RGB-D:', SUN_ROOT)
print('Output:', RUN_DIR)
print('Resume:', RESUME_FROM if RESUME_FROM.is_file() else 'tidak ada checkpoint; mulai dari awal')

ModuleNotFoundError: No module named 'google.colab'

## Arsitektur model

Sel berikut menanamkan blueprint custom laporan ke runtime Colab sebagai modul lokal untuk dipakai oleh trainer di sel berikutnya.

In [ ]:
%%writefile /content/arsitektur.py
"""Custom YOLO26n-Depth blueprint from report Sections 3.1 and 3.2."""

from __future__ import annotations

import argparse

import torch
from torch import Tensor, nn
from torch.nn import functional as F


class ConvBlock(nn.Module):
    """Convolution, batch normalization, and SiLU activation."""

    def __init__(self, in_ch: int, out_ch: int, k: int = 3, s: int = 1, p: int | None = None, g: int = 1) -> None:
        super().__init__()
        padding = k // 2 if p is None else p
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=padding, groups=g, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU()

    def forward(self, x: Tensor) -> Tensor:
        return self.act(self.bn(self.conv(x)))


def _scaled_channels(base_channels: int, width_mult: float, minimum: int, max_channels: int) -> int:
    return max(minimum, min(int(base_channels * width_mult), max_channels))


class Bottleneck(nn.Module):
    def __init__(self, ch: int) -> None:
        super().__init__()
        self.cv1 = ConvBlock(ch, ch, k=3, s=1, p=1)
        self.cv2 = ConvBlock(ch, ch, k=3, s=1, p=1)

    def forward(self, x: Tensor) -> Tensor:
        return x + self.cv2(self.cv1(x))


class C3k2(nn.Module):
    def __init__(self, c1: int, c2: int, n: int = 1) -> None:
        super().__init__()
        if n < 1:
            raise ValueError("C3k2 repeat count must be positive.")
        hidden = max(1, c2 // 2)
        self.cv1 = ConvBlock(c1, hidden, k=1, s=1, p=0)
        self.cv2 = ConvBlock(c1, hidden, k=1, s=1, p=0)
        self.blocks = nn.Sequential(*(Bottleneck(hidden) for _ in range(n)))
        self.cv3 = ConvBlock(2 * hidden, c2, k=1, s=1, p=0)

    def forward(self, x: Tensor) -> Tensor:
        return self.cv3(torch.cat((self.cv1(x), self.blocks(self.cv2(x))), dim=1))


class C2PSA(nn.Module):
    def __init__(self, c1: int, c2: int) -> None:
        super().__init__()
        self.cv1 = ConvBlock(c1, c2, k=1, s=1, p=0)
        hidden = max(1, c2 // 4)
        self.attn = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c2, hidden, 1), nn.SiLU(), nn.Conv2d(hidden, c2, 1), nn.Sigmoid())

    def forward(self, x: Tensor) -> Tensor:
        features = self.cv1(x)
        return features * self.attn(features)


class SPPF(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 5) -> None:
        super().__init__()
        hidden_ch = max(1, in_ch // 2)
        self.cv1 = ConvBlock(in_ch, hidden_ch, k=1, s=1, p=0)
        self.cv2 = ConvBlock(hidden_ch * 4, out_ch, k=1, s=1, p=0)
        self.pool = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x: Tensor) -> Tensor:
        x = self.cv1(x)
        y1 = self.pool(x)
        y2 = self.pool(y1)
        y3 = self.pool(y2)
        return self.cv2(torch.cat((x, y1, y2, y3), dim=1))


class Backbone(nn.Module):
    def __init__(self, c_stem: int, channels: tuple[int, int, int, int], n_repeat: int) -> None:
        super().__init__()
        c_p2, c_p3, c_p4, c_p5 = channels
        self.stem = ConvBlock(3, c_stem, k=3, s=2, p=1)
        self.l2 = nn.Sequential(ConvBlock(c_stem, c_p2, k=3, s=2, p=1), C3k2(c_p2, c_p2, n=n_repeat))
        self.l3 = ConvBlock(c_p2, c_p3, k=3, s=2, p=1)
        self.l4 = C3k2(c_p3, c_p3, n=n_repeat)
        self.l5 = ConvBlock(c_p3, c_p4, k=3, s=2, p=1)
        self.l6 = C3k2(c_p4, c_p4, n=n_repeat)
        self.l7 = ConvBlock(c_p4, c_p5, k=3, s=2, p=1)
        self.l8 = C3k2(c_p5, c_p5, n=n_repeat)
        self.l9 = SPPF(c_p5, c_p5, k=5)
        self.l10 = C2PSA(c_p5, c_p5)

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        x = self.l2(self.stem(x))
        p3 = self.l4(self.l3(x))
        p4 = self.l6(self.l5(p3))
        p5 = self.l10(self.l9(self.l8(self.l7(p4))))
        return p3, p4, p5


def _upsample_like(x: Tensor, reference: Tensor) -> Tensor:
    return F.interpolate(x, size=reference.shape[-2:], mode="nearest")


class Neck(nn.Module):
    def __init__(self, channels: tuple[int, int, int], n_repeat: int) -> None:
        super().__init__()
        c_p3, c_p4, c_p5 = channels
        self.n13 = C3k2(c_p5 + c_p4, c_p4, n=n_repeat)
        self.n16 = C3k2(c_p4 + c_p3, c_p3, n=n_repeat)
        self.d17 = ConvBlock(c_p3, c_p3, k=3, s=2, p=1)
        self.n19 = C3k2(c_p3 + c_p4, c_p4, n=n_repeat)
        self.d20 = ConvBlock(c_p4, c_p4, k=3, s=2, p=1)
        self.n22 = C3k2(c_p4 + c_p5, c_p5, n=n_repeat)

    def forward(self, p3: Tensor, p4: Tensor, p5: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        p4_fused = self.n13(torch.cat((_upsample_like(p5, p4), p4), dim=1))
        p3_out = self.n16(torch.cat((_upsample_like(p4_fused, p3), p3), dim=1))
        p4_out = self.n19(torch.cat((self.d17(p3_out), p4_fused), dim=1))
        p5_out = self.n22(torch.cat((self.d20(p4_out), p5), dim=1))
        return p3_out, p4_out, p5_out


class DetectionHead(nn.Module):
    def __init__(self, in_ch: int, num_classes: int = 4) -> None:
        super().__init__()
        if num_classes < 1:
            raise ValueError("num_classes must be positive.")
        self.reg = nn.Sequential(ConvBlock(in_ch, in_ch, k=3, s=1, p=1), nn.Conv2d(in_ch, 4, kernel_size=1))
        self.cls = nn.Sequential(ConvBlock(in_ch, in_ch, k=3, s=1, p=1), nn.Conv2d(in_ch, num_classes, kernel_size=1))

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor]:
        return self.reg(x), self.cls(x)


class DepthHead(nn.Module):
    def __init__(self, channels: tuple[int, int, int]) -> None:
        super().__init__()
        self.conv_p3 = ConvBlock(channels[0], 32, k=1, s=1, p=0)
        self.conv_p4 = ConvBlock(channels[1], 32, k=1, s=1, p=0)
        self.conv_p5 = ConvBlock(channels[2], 32, k=1, s=1, p=0)
        self.refine = nn.Sequential(ConvBlock(96, 32, k=3, s=1, p=1), ConvBlock(32, 16, k=3, s=1, p=1), nn.Conv2d(16, 1, kernel_size=1), nn.Sigmoid())

    def forward(self, features: tuple[Tensor, Tensor, Tensor], target_size: tuple[int, int]) -> Tensor:
        p3, p4, p5 = features
        projected = [self.conv_p3(p3), self.conv_p4(p4), self.conv_p5(p5)]
        upsampled = [F.interpolate(feature, size=target_size, mode="bilinear", align_corners=False) for feature in projected]
        return self.refine(torch.cat(upsampled, dim=1))


ARCHITECTURE_ID = "report-blueprint-3.1-3.2-v1"


class YOLO26nDepth(nn.Module):
    def __init__(self, num_classes: int = 4, width_mult: float = 0.25, depth_mult: float = 0.50, max_channels: int = 1024) -> None:
        super().__init__()
        if num_classes < 1:
            raise ValueError("num_classes must be positive.")
        if width_mult <= 0 or depth_mult <= 0 or max_channels < 1:
            raise ValueError("Scaling factors and max_channels must be positive.")
        c_stem = _scaled_channels(64, width_mult, 32, max_channels)
        c_p2 = _scaled_channels(128, width_mult, 32, max_channels)
        c_p3 = _scaled_channels(256, width_mult, 64, max_channels)
        c_p4 = _scaled_channels(512, width_mult, 128, max_channels)
        c_p5 = _scaled_channels(1024, width_mult, 256, max_channels)
        n_repeat = max(1, round(2 * depth_mult))
        self.backbone = Backbone(c_stem, (c_p2, c_p3, c_p4, c_p5), n_repeat)
        self.neck = Neck((c_p3, c_p4, c_p5), n_repeat)
        self.detect_p3 = DetectionHead(c_p3, num_classes)
        self.detect_p4 = DetectionHead(c_p4, num_classes)
        self.detect_p5 = DetectionHead(c_p5, num_classes)
        self.depth_head = DepthHead((c_p3, c_p4, c_p5))

    def forward(self, x: Tensor) -> dict[str, Tensor]:
        if x.ndim != 4 or x.shape[1] != 3:
            raise ValueError("Input must have shape [batch, 3, height, width].")
        if x.shape[-2] < 32 or x.shape[-1] < 32:
            raise ValueError("Input height and width must each be at least 32.")
        if x.shape[-2] % 32 or x.shape[-1] % 32:
            raise ValueError("Input height and width must be divisible by 32.")
        p3, p4, p5 = self.backbone(x)
        p3_out, p4_out, p5_out = self.neck(p3, p4, p5)
        p3_reg, p3_cls = self.detect_p3(p3_out)
        p4_reg, p4_cls = self.detect_p4(p4_out)
        p5_reg, p5_cls = self.detect_p5(p5_out)
        depth_map = self.depth_head((p3_out, p4_out, p5_out), x.shape[-2:])
        return {"p3_bbox": p3_reg, "p3_cls": p3_cls, "p4_bbox": p4_reg, "p4_cls": p4_cls, "p5_bbox": p5_reg, "p5_cls": p5_cls, "depth": depth_map}


def build_model(num_classes: int = 4, width_mult: float = 0.25, depth_mult: float = 0.50) -> YOLO26nDepth:
    return YOLO26nDepth(num_classes=num_classes, width_mult=width_mult, depth_mult=depth_mult, max_channels=1024)


def _verify() -> None:
    model = build_model().eval()
    images = torch.zeros(1, 3, 640, 640)
    with torch.inference_mode():
        output = model(images)
    print(f"Input: {tuple(images.shape)}")
    for key in ("p3_bbox", "p3_cls", "p4_bbox", "p4_cls", "p5_bbox", "p5_cls", "depth"):
        print(f"{key}: {tuple(output[key].shape)}")
    print(f"Parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Verify the report-aligned YOLO26n-Depth architecture.")
    parser.add_argument("--verify", action="store_true")
    args = parser.parse_args()
    if not args.verify:
        parser.error("Use --verify to run the architecture smoke test.")
    _verify()

In [ ]:
from dataclasses import dataclass
import json
import math
import random
from pathlib import Path

import cv2
import numpy as np
import torch
from torch import Tensor
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from arsitektur import ARCHITECTURE_ID, build_model


@dataclass(frozen=True)
class SunRecord:
    image_path: Path
    depth_path: Path
    boxes: tuple[tuple[float, float, float, float, int], ...]
    group: str


def _scene_image(scene):
    folder = scene / 'image'
    if not folder.is_dir():
        return None
    return next((path for path in sorted(folder.iterdir()) if path.suffix.lower() in {'.jpg', '.jpeg', '.png'}), None)


def _scene_depth(scene):
    for folder_name in ('depth', 'depth_bfx'):
        folder = scene / folder_name
        if folder.is_dir():
            matches = sorted(folder.glob('*.png'))
            if matches:
                return matches[0]
    return None


def _object_names(annotation):
    result = []
    for item in annotation.get('objects', []):
        value = item.get('name') if isinstance(item, dict) else item
        result.append(value.split(':')[0].strip().lower() if isinstance(value, str) else None)
    return result


def _annotation_polygons(annotation):
    frames = annotation.get('frames', [])
    if isinstance(frames, dict):
        frames = [frames]
    if not frames or not isinstance(frames[0], dict):
        return []
    polygons = frames[0].get('polygon', [])
    return [polygons] if isinstance(polygons, dict) else [item for item in polygons if isinstance(item, dict)]


def _sun_object_base(paths):
    over_count = zero_count = 0
    for path in paths[:200]:
        try:
            annotation = json.loads(path.read_text(encoding='utf-8'))
        except (OSError, json.JSONDecodeError):
            continue
        object_count = len(_object_names(annotation))
        for polygon in _annotation_polygons(annotation):
            object_id = polygon.get('object')
            if isinstance(object_id, int):
                over_count += int(object_id >= object_count)
                zero_count += int(object_id == 0)
    return 1 if over_count > 0 and zero_count == 0 else 0


def _scene_group(scene, root):
    relative = scene.relative_to(root).as_posix()
    parent, name = relative.rsplit('/', 1) if '/' in relative else ('', relative)
    digits = ''.join(character for character in reversed(name) if character.isdigit())[::-1]
    if digits and ('nyudata' in parent.lower() or 'b3dodata' in parent.lower()):
        return f'{parent}/block{int(digits) // 100}'
    return parent or relative


def discover_sun(root, classes):
    root = Path(root).expanduser()
    annotations = sorted(root.rglob('annotation2Dfinal/index.json'))
    if not annotations:
        raise FileNotFoundError(f'Tidak menemukan annotation2Dfinal/index.json di {root}')
    class_to_id = {name.casefold(): index for index, name in enumerate(classes)}
    object_base = _sun_object_base(annotations)
    records = []
    invalid = 0
    for annotation_path in annotations:
        scene = annotation_path.parent.parent
        image_path, depth_path = _scene_image(scene), _scene_depth(scene)
        if image_path is None or depth_path is None:
            continue
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            continue
        try:
            annotation = json.loads(annotation_path.read_text(encoding='utf-8'))
        except (OSError, json.JSONDecodeError):
            invalid += 1
            continue
        names = _object_names(annotation)
        height, width = image.shape[:2]
        boxes = []
        for polygon in _annotation_polygons(annotation):
            object_id = polygon.get('object')
            if not isinstance(object_id, int):
                continue
            object_index = object_id - object_base
            if not 0 <= object_index < len(names):
                continue
            class_id = class_to_id.get(names[object_index] or '')
            if class_id is None:
                continue
            try:
                xs = np.asarray(polygon.get('x', []), dtype=np.float32).reshape(-1)
                ys = np.asarray(polygon.get('y', []), dtype=np.float32).reshape(-1)
            except (TypeError, ValueError):
                continue
            if len(xs) < 2 or len(xs) != len(ys):
                continue
            x1, y1 = max(float(xs.min()), 0.0), max(float(ys.min()), 0.0)
            x2, y2 = min(float(xs.max()), width - 1.0), min(float(ys.max()), height - 1.0)
            if x2 - x1 >= 4 and y2 - y1 >= 4:
                boxes.append((x1, y1, x2, y2, class_id))
        records.append(SunRecord(image_path, depth_path, tuple(boxes), _scene_group(scene, root)))
    print(f'SUN RGB-D records: {len(records)}; anotasi invalid dilewati: {invalid}')
    if len(records) < 2:
        raise ValueError('Dataset SUN RGB-D perlu memiliki minimal dua scene valid.')
    return records


def _read_depth(path, rotate, depth_max_m):
    raw = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if raw is None:
        raise OSError(f'Tidak dapat membaca depth map: {path}')
    if raw.ndim == 3:
        raw = raw[..., 0]
    values = raw.astype(np.uint32)
    if rotate:
        values = ((values >> 3) | (values << 13)) & 0xFFFF
    depth = values.astype(np.float32) / 1000.0
    depth[(depth <= 0) | (depth > depth_max_m)] = 0
    return depth


def _detect_depth_rotation(records):
    votes = used = 0
    for record in records[:64]:
        raw = cv2.imread(str(record.depth_path), cv2.IMREAD_UNCHANGED)
        if raw is None:
            continue
        raw = raw.astype(np.uint16)
        rotated = (((raw.astype(np.uint32) >> 3) | (raw.astype(np.uint32) << 13)) & 0xFFFF).astype(np.uint16)
        raw_values, rotated_values = raw[raw > 0], rotated[rotated > 0]
        if raw_values.size < 100 or not rotated_values.size:
            continue
        raw_median = float(np.median(raw_values)) / 1000.0
        rotated_median = float(np.median(rotated_values)) / 1000.0
        votes += int(0.3 < rotated_median < 8 and not 0.3 < raw_median < 8)
        used += 1
    return used > 0 and votes >= used / 2


def _prepare(image, depth, boxes, image_size, training, rng, depth_max_m):
    target_h, target_w = image_size
    source_h, source_w = image.shape[:2]
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 5)
    if training and rng.random() < 0.5:
        image = np.ascontiguousarray(image[:, ::-1])
        depth = np.ascontiguousarray(depth[:, ::-1])
        if len(boxes):
            left, right = boxes[:, 0].copy(), boxes[:, 2].copy()
            boxes[:, 0], boxes[:, 2] = source_w - right, source_w - left
    scale = min(target_w / source_w, target_h / source_h)
    resized_w = max(1, min(round(source_w * scale), target_w))
    resized_h = max(1, min(round(source_h * scale), target_h))
    pad_x, pad_y = (target_w - resized_w) // 2, (target_h - resized_h) // 2
    resized_image = cv2.resize(image, (resized_w, resized_h), interpolation=cv2.INTER_LINEAR)
    resized_depth = cv2.resize(depth, (resized_w, resized_h), interpolation=cv2.INTER_NEAREST)
    canvas = np.full((target_h, target_w, 3), 114, dtype=np.uint8)
    depth_canvas = np.zeros((target_h, target_w), dtype=np.float32)
    canvas[pad_y:pad_y + resized_h, pad_x:pad_x + resized_w] = resized_image
    depth_canvas[pad_y:pad_y + resized_h, pad_x:pad_x + resized_w] = resized_depth
    if len(boxes):
        boxes[:, (0, 2)] = (boxes[:, (0, 2)] * scale + pad_x).clip(pad_x, pad_x + resized_w)
        boxes[:, (1, 3)] = (boxes[:, (1, 3)] * scale + pad_y).clip(pad_y, pad_y + resized_h)
    image_tensor = torch.from_numpy(canvas.copy()).permute(2, 0, 1).float() / 255.0
    depth_tensor = torch.from_numpy(depth_canvas / depth_max_m).clamp_(0.0, 1.0)
    return image_tensor, depth_tensor, torch.from_numpy(boxes)


class SunDataset(Dataset):
    def __init__(self, records, image_size, training, depth_max_m, rotate, seed):
        self.records, self.image_size, self.training = records, image_size, training
        self.depth_max_m, self.rotate, self.seed, self.epoch = depth_max_m, rotate, seed, 0

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = cv2.imread(str(record.image_path), cv2.IMREAD_COLOR)
        if image is None:
            raise OSError(f'Tidak dapat membaca image: {record.image_path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        depth = _read_depth(record.depth_path, self.rotate, self.depth_max_m)
        if depth.shape != image.shape[:2]:
            depth = cv2.resize(depth, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
        rng = random.Random(self.seed + self.epoch * len(self) + index)
        return _prepare(image, depth, record.boxes, self.image_size, self.training, rng, self.depth_max_m)


def _collate(batch):
    images, depths, boxes = zip(*batch)
    return torch.stack(images), torch.stack(depths), list(boxes)


def _candidate_levels(box_w, box_h):
    side = math.sqrt(max(box_w * box_h, 0.0))
    preferred = 0 if side < 64 else 1 if side < 160 else 2
    return sorted(range(3), key=lambda level: (abs(level - preferred), level))


def _build_targets(outputs, boxes_per_image, device):
    levels = ((8, 'p3'), (16, 'p4'), (32, 'p5'))
    class_targets, box_targets, masks = [], [], []
    for _, prefix in levels:
        logits = outputs[f'{prefix}_cls']
        class_targets.append(torch.zeros_like(logits))
        box_targets.append(torch.zeros_like(outputs[f'{prefix}_bbox']))
        masks.append(torch.zeros(logits.shape[0], logits.shape[2], logits.shape[3], dtype=torch.bool, device=device))
    for batch_index, batch_boxes in enumerate(boxes_per_image):
        for x1, y1, x2, y2, class_id in batch_boxes.tolist():
            width, height = max(x2 - x1, 1.0), max(y2 - y1, 1.0)
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            assignment = None
            for level in _candidate_levels(width, height):
                stride, prefix = levels[level]
                grid_h, grid_w = outputs[f'{prefix}_cls'].shape[-2:]
                col, row = min(int(cx / stride), grid_w - 1), min(int(cy / stride), grid_h - 1)
                candidates = [(r, c) for r in range(max(0, row - 2), min(grid_h, row + 3)) for c in range(max(0, col - 2), min(grid_w, col + 3))]
                candidates.sort(key=lambda cell: ((cell[1] + 0.5) * stride - cx) ** 2 + ((cell[0] + 0.5) * stride - cy) ** 2)
                for r, c in candidates:
                    if not masks[level][batch_index, r, c]:
                        assignment = (level, r, c, stride)
                        break
                if assignment:
                    break
            if assignment is None:
                raise RuntimeError('Could not assign a target box to a free grid cell.')
            level, row, col, stride = assignment
            masks[level][batch_index, row, col] = True
            class_targets[level][batch_index, int(class_id), row, col] = 1.0
            box_targets[level][batch_index, :, row, col] = torch.tensor((cx / stride - (col + 0.5), cy / stride - (row + 0.5), math.log(max(width / stride, 1e-3)), math.log(max(height / stride, 1e-3))), device=device, dtype=box_targets[level].dtype)
    return class_targets, box_targets, masks


def _loss(outputs, target_depth, boxes_per_image, device):
    class_targets, box_targets, masks = _build_targets(outputs, boxes_per_image, device)
    cls_loss = outputs['p3_cls'].new_zeros(())
    box_parts = []
    for level, (_, prefix) in enumerate(((8, 'p3'), (16, 'p4'), (32, 'p5'))):
        logits = outputs[f'{prefix}_cls']
        probability = logits.sigmoid()
        cross_entropy = F.binary_cross_entropy_with_logits(logits, class_targets[level], reduction='none')
        p_t = probability * class_targets[level] + (1 - probability) * (1 - class_targets[level])
        alpha_t = 0.25 * class_targets[level] + 0.75 * (1 - class_targets[level])
        cls_loss = cls_loss + (alpha_t * (1 - p_t).square() * cross_entropy).mean() / 3
        if masks[level].any():
            predicted = outputs[f'{prefix}_bbox'].permute(0, 2, 3, 1)[masks[level]]
            target = box_targets[level].permute(0, 2, 3, 1)[masks[level]]
            box_parts.append(F.smooth_l1_loss(predicted, target, beta=1.0))
    box_loss = torch.stack(box_parts).mean() if box_parts else outputs['p3_bbox'].sum() * 0
    predicted_depth = outputs['depth'][:, 0]
    valid = target_depth > 0
    depth_loss = F.smooth_l1_loss(predicted_depth[valid], target_depth[valid], beta=0.05) if valid.any() else predicted_depth.sum() * 0
    total = cls_loss + 5.0 * box_loss + depth_loss
    return total, {'classification': cls_loss, 'box': box_loss, 'depth': depth_loss}


def _split_records(records, val_fraction, seed):
    groups = {}
    for record in records:
        groups.setdefault(record.group, []).append(record)
    keys = list(groups)
    random.Random(seed).shuffle(keys)
    if len(keys) < 2:
        raise ValueError('Need at least two SUN RGB-D scene groups.')
    target = max(1, round(len(records) * val_fraction))
    val_keys, count = [], 0
    for key in keys:
        if count >= target or len(val_keys) == len(keys) - 1:
            break
        val_keys.append(key)
        count += len(groups[key])
    val_set = set(val_keys)
    return ([record for record in records if record.group not in val_set], [record for record in records if record.group in val_set])


class ActivityIndicator:
    def __init__(self, message, interval=0.15):
        self.message, self.interval = message, interval
        self.lock, self.stop = threading.Lock(), threading.Event()
        self.thread = threading.Thread(target=self._animate, daemon=True)

    def _animate(self):
        frames, index = '|/-\\', 0
        while not self.stop.wait(self.interval):
            with self.lock:
                message = self.message
            print(f'\\r[{frames[index % len(frames)]}] {message}', end='', flush=True)
            index += 1

    def update(self, message):
        with self.lock:
            self.message = message

    def __enter__(self):
        self.thread.start()
        return self

    def __exit__(self, exception_type, *_):
        self.stop.set()
        self.thread.join()
        with self.lock:
            message = self.message
        status = 'stopped' if exception_type is not None else 'done'
        print(f'\\r[{status}] {message}', flush=True)


def train_colab(resume_from=None):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    records = discover_sun(SUN_ROOT, CLASSES)
    rotate = _detect_depth_rotation(records)
    print('Depth rotation:', 'on' if rotate else 'off', '| device:', device)
    train_records, val_records = _split_records(records, 0.1, SEED)
    train_ds = SunDataset(train_records, IMAGE_SIZE, True, DEPTH_MAX_M, rotate, SEED)
    val_ds = SunDataset(val_records, IMAGE_SIZE, False, DEPTH_MAX_M, rotate, SEED)
    loader_options = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda', collate_fn=_collate)
    train_loader = DataLoader(train_ds, shuffle=True, **loader_options)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_options)
    model = build_model(num_classes=len(CLASSES)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    start_epoch, best_loss = 0, float('inf')
    if resume_from is not None and Path(resume_from).is_file():
        checkpoint = torch.load(resume_from, map_location=device, weights_only=True)
        if checkpoint.get('kind') != 'arsitektur_multitask':
            raise ValueError('Resume checkpoint is not a report-blueprint checkpoint.')
        if checkpoint.get('architecture_id') not in (None, ARCHITECTURE_ID):
            raise ValueError('Resume checkpoint architecture does not match this notebook.')
        if checkpoint['classes'] != list(CLASSES) or tuple(checkpoint['imgsz']) != tuple(IMAGE_SIZE):
            raise ValueError('Resume checkpoint classes or image size do not match notebook settings.')
        model.load_state_dict(checkpoint['model_state_dict'], strict=True)
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = int(checkpoint['epoch'])
        best_loss = float(checkpoint['best_val_loss'])
        print(f'Resuming after epoch {start_epoch}.')
    for epoch in range(start_epoch, EPOCHS):
        train_ds.epoch = epoch
        model.train()
        train_losses = []
        train_iterator = iter(train_loader)
        with ActivityIndicator(f'Epoch {epoch + 1}/{EPOCHS}: training') as activity:
            for batch_index in range(len(train_loader)):
                activity.update(f'Epoch {epoch + 1}/{EPOCHS}: loading train batch {batch_index + 1}/{len(train_loader)}')
                images, depths, boxes = next(train_iterator)
                activity.update(f'Epoch {epoch + 1}/{EPOCHS}: training batch {batch_index + 1}/{len(train_loader)}')
                images, depths = images.to(device), depths.to(device)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                    outputs = model(images)
                    loss, parts = _loss(outputs, depths, boxes, device)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f'Non-finite training loss at epoch {epoch + 1}.')
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
                scaler.step(optimizer)
                scaler.update()
                train_losses.append(float(loss.detach()))
        model.eval()
        val_losses, abs_error, valid_pixels = [], 0.0, 0
        with torch.inference_mode():
            val_iterator = iter(val_loader)
            with ActivityIndicator(f'Epoch {epoch + 1}/{EPOCHS}: validation') as activity:
                for batch_index in range(len(val_loader)):
                    activity.update(f'Epoch {epoch + 1}/{EPOCHS}: loading validation batch {batch_index + 1}/{len(val_loader)}')
                    images, depths, boxes = next(val_iterator)
                    activity.update(f'Epoch {epoch + 1}/{EPOCHS}: validating batch {batch_index + 1}/{len(val_loader)}')
                    images, depths = images.to(device), depths.to(device)
                    outputs = model(images)
                    loss, _ = _loss(outputs, depths, boxes, device)
                    val_losses.append(float(loss))
                    valid = depths > 0
                    if valid.any():
                        abs_error += float((outputs['depth'][:, 0][valid] - depths[valid]).abs().sum())
                        valid_pixels += int(valid.sum())
        scheduler.step()
        mean_val = sum(val_losses) / len(val_losses)
        mae_m = abs_error / valid_pixels * DEPTH_MAX_M if valid_pixels else float('nan')
        print(f'Epoch {epoch + 1}/{EPOCHS}: train_loss={sum(train_losses) / len(train_losses):.4f}, val_loss={mean_val:.4f}, depth_MAE={mae_m:.3f} m')
        is_best = mean_val < best_loss
        if is_best:
            best_loss = mean_val
        checkpoint = {'kind': 'arsitektur_multitask', 'architecture_id': ARCHITECTURE_ID, 'epoch': epoch + 1, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(), 'best_val_loss': best_loss, 'classes': list(CLASSES), 'imgsz': list(IMAGE_SIZE), 'depth_max_m': DEPTH_MAX_M, 'width_mult': 0.25, 'depth_mult': 0.5}
        RUN_DIR.mkdir(parents=True, exist_ok=True)
        torch.save(checkpoint, RUN_DIR / 'last.pt')
        if is_best:
            torch.save(checkpoint, RUN_DIR / 'best.pt')
    print('Training selesai. Checkpoint:', RUN_DIR / 'best.pt')

print('Trainer SUN RGB-D siap. Jalankan sel training setelah konfigurasi.')

## Pipeline training SUN RGB-D

Sel berikut memuat discovery dataset, preprocessing, target/loss, progress batch, resume, dan penyimpanan checkpoint. Kode tidak mengambil file trainer dari luar notebook.

## Cek runtime Colab

PyTorch biasanya sudah disediakan bersama CUDA oleh Colab. Cell ini tidak mengganti/meng-install ulang PyTorch agar versi CUDA Colab tidak rusak. Jika `CUDA tersedia` bernilai `False`, ubah runtime ke GPU lalu restart runtime.

In [ ]:
import cv2
import numpy as np
import torch

print('PyTorch:', torch.__version__)
print('OpenCV:', cv2.__version__)
print('NumPy:', np.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('PERINGATAN: training berjalan di CPU dan akan jauh lebih lambat.')

## Konfigurasi training

Atur jumlah epoch, batch, kelas, ukuran input, dan checkpoint resume. `SUN_ROOT` serta `RUN_DIR` sudah ditentukan pada sel Drive.

In [ ]:
SMOKE_TEST = False  # True untuk 1 epoch; tetap memakai dataset penuh
if SMOKE_TEST:
    EPOCHS = 1

print({
    'classes': CLASSES,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'image_size': IMAGE_SIZE,
    'depth_max_m': DEPTH_MAX_M,
    'sun_root': str(SUN_ROOT),
    'resume_from': str(RESUME_FROM) if RESUME_FROM.is_file() else None,
})

## Jalankan training

Pastikan `SUN_ROOT` pada sel Drive menunjuk ke folder SUN RGB-D yang sudah diekstrak. Checkpoint `best.pt` dan `last.pt` akan disimpan langsung ke `RUN_DIR` di Drive. Jika `RESUME_FROM` tersedia, training melanjutkan dari checkpoint tersebut.

In [ ]:
if not SUN_ROOT.is_dir():
    raise FileNotFoundError(f'Folder SUN RGB-D tidak ditemukan: {SUN_ROOT}')

resume_path = RESUME_FROM if RESUME_FROM.is_file() else None
train_colab(resume_path)

## Periksa checkpoint terbaik

Depth disimpan sebagai nilai relatif `meter / depth_max_m` sesuai sigmoid di depth head. Checkpoint ini berasal dari `arsitektur.py`; checkpoint ini bukan `multitask_best.pt` dan belum digunakan aplikasi kamera.

In [ ]:
BEST_CHECKPOINT = RUN_DIR / 'best.pt'
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Checkpoint terbaik belum ada: {BEST_CHECKPOINT}')
checkpoint = torch.load(BEST_CHECKPOINT, map_location='cpu', weights_only=True)
print('Checkpoint:', BEST_CHECKPOINT)
print('Kind:', checkpoint['kind'])
print('Epoch:', checkpoint['epoch'])
print('Kelas:', checkpoint['classes'])
print('Input (H, W):', checkpoint['imgsz'])
print('Depth max (m):', checkpoint['depth_max_m'])
print('Best validation loss:', checkpoint['best_val_loss'])